# 03 — Claim Frequency: Baseline and Poisson GLM

**Phase 5**, steps 1-2 of the plan:

1. A trivial baseline (portfolio-wide mean) and a first Poisson GLM with `log(Exposure)` as an offset, fit on the training split only.
2. Overdispersion diagnostics - does the Poisson assumption (`Var(N) = E(N)`) actually hold for this fitted model? That decides whether a Negative Binomial challenger is warranted in a later step.

Every feature used here comes from `reports/feature_dictionary.md` (Phase 4). All fitting happens on `split == "train"` only - validation and test are not touched in this notebook.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.frequency import (
    FREQUENCY_FORMULA,
    baseline_frequency,
    deviance_dispersion_ratio,
    dispersion_ratio,
    fit_poisson_glm,
    predict_frequency,
)

model_table = pd.read_parquet("../data/processed/model_table.parquet")
train = model_table[model_table["split"] == "train"].copy()
print(f"train rows: {len(train):,}")

train rows: 474,609


## 1. Baseline model

In [2]:
baseline = baseline_frequency(train)
print(f"baseline (portfolio-mean) frequency: {baseline:.5f} claims/policy-year")

baseline (portfolio-mean) frequency: 0.10055 claims/policy-year


This is the floor: predict the same 0.1006 claims/policy-year for every single policy, regardless of driver age, BonusMalus, or anything else. Any model that can't beat this isn't learning anything useful from the rating factors.

## 2. Poisson GLM with an exposure offset

In [3]:
print(FREQUENCY_FORMULA)

ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + AreaOrdinal + LogDensity + VehGasBinary + C(RegionGrouped) + C(VehBrandGrouped) + VehPower


In [4]:
result = fit_poisson_glm(train)
print(f"converged: {result.converged}")
print(f"parameters: {len(result.params)}")
print(f"deviance: {result.deviance:.1f}")

converged: True
parameters: 48
deviance: 148707.6


### Sanity check: does total predicted claims match total observed claims?

A Poisson GLM fit by maximum likelihood with a log link has a guaranteed property on its own training data: total predicted claims must equal total observed claims exactly. This is a useful sanity check that the model was fit and is being read correctly - and, as it turned out while building this, catching a real mistake in how predictions were being computed at all.

In [5]:
predicted_total = predict_frequency(result, train).sum()
observed_total = train["ClaimNb"].sum()
print(f"predicted total claims: {predicted_total:,.2f}")
print(f"observed total claims:  {observed_total:,}")

predicted total claims: 25,227.00
observed total claims:  25,227


### A bug caught here, not glossed over

The first version of this check used `result.predict(train)` directly - the "obvious" way to get predictions from a fitted statsmodels model. It gave a predicted total of **54,154** against an observed total of **25,227** - overstated by more than 2x, with no error or warning raised. The cause: statsmodels' `.predict()` silently drops the offset unless it is passed again explicitly, so it implicitly assumed every policy had a full year of exposure. Since the average policy in this data has only about half a year of exposure, that assumption alone explains most of the 2x inflation.

`predict_frequency()` (in `src/auto_pricing/frequency.py`) always re-supplies the offset, so this mistake is now structurally impossible to repeat - and a test (`tests/test_frequency.py`) pins down the correct behavior so it can't silently regress.

In [6]:
# The mistake, shown deliberately for contrast - never use this form:
naive_wrong_total = result.predict(train).sum()
print(f"naive result.predict(train), offset silently dropped: {naive_wrong_total:,.2f}")
print(f"correct predict_frequency(result, train):              {predicted_total:,.2f}")
print(f"observed:                                               {observed_total:,}")

naive result.predict(train), offset silently dropped: 54,153.85
correct predict_frequency(result, train):              25,227.00
observed:                                               25,227


### A look at some fitted relativities

In [7]:
result.params.filter(like="BonusMalusBand").apply(lambda x: round(x, 3)).to_frame("coefficient").assign(
    relativity=lambda d: np.exp(d["coefficient"]).round(3)
)

,coefficient,relativity
C(BonusMalusBand)[T.51-59],0.305,1.357
C(BonusMalusBand)[T.60-79],0.646,1.908
C(BonusMalusBand)[T.80-99],0.787,2.197
C(BonusMalusBand)[T.100-129],1.473,4.362
C(BonusMalusBand)[T.130+],1.860,6.424


Coefficients from a Poisson GLM with a log link are on a *multiplicative* scale: `exp(coefficient)` gives a relativity relative to the reference category (here, `BonusMalus = 50`, the best score, since it's the first band). A relativity of 1.91 for `60-79` means that band's expected frequency is 91% higher than the reference band, all else held equal - consistent with Phase 3's finding of a strong, monotonic BonusMalus effect. Full interpretation of every factor is deferred to Phase 5's interpretation step.

## 3. Overdispersion diagnostics

**Background:** a Poisson distribution makes one strong assumption - its variance always equals its mean (`Var(N) = E(N)`). Real claims data often doesn't behave this way: some policies are riskier than the model's inputs capture, in ways that add extra spread beyond what a pure Poisson process would produce. This is called **overdispersion**, and it's worth checking directly rather than assuming - the plan is explicit that a more complex model (Negative Binomial) should only be added if the evidence actually calls for it.

In [8]:
print(f"Pearson dispersion (chi2/df):   {dispersion_ratio(result):.4f}")
print(f"Deviance dispersion (dev/df):   {deviance_dispersion_ratio(result):.4f}")

Pearson dispersion (chi2/df):   2.3080
Deviance dispersion (dev/df):   0.3134


These two diagnostics disagree sharply - 2.31 (suggesting real overdispersion) versus 0.31 (suggesting the opposite). That disagreement itself needs explaining before trusting either number.

In [9]:
mu = result.fittedvalues
print(mu.describe())
print()
print(f"share of policies with fitted mu < 1: {(mu < 1).mean():.4%}")
print(f"share of policies with fitted mu < 0.2: {(mu < 0.2).mean():.4%}")

count    474609.000000
mean          0.053153
std           0.049212
min           0.000072
25%           0.017474
50%           0.045828
75%           0.074446
max           1.227878
dtype: float64

share of policies with fitted mu < 1: 99.9977%
share of policies with fitted mu < 0.2: 98.4511%


**Why the two diagnostics disagree, and which to trust:** the deviance statistic's chi-squared approximation - the basis for treating `deviance/df` as a dispersion check - requires fitted means (`mu`) that aren't too small. Here, 99.998% of policies have a fitted mean under 1, and the median is just 0.046: this is a genuinely rare-event portfolio, exactly the regime where that approximation is known to break down. The Pearson chi-square statistic doesn't carry the same requirement and is the diagnostic generally trusted for low-frequency insurance data in actuarial practice. **Conclusion: the Pearson dispersion ratio (2.31) is the one to act on. It indicates material overdispersion** - more than double the variance a plain Poisson model allows for - which is real evidence for considering a Negative Binomial challenger, per the plan's own criterion (add complexity only when diagnostics justify it, not by default).

### A quick, independent sanity check: raw data, no model

Before trusting a model-based diagnostic, it's worth checking the rawest possible version of the same question.

In [10]:
raw_mean = train["ClaimNb"].mean()
raw_var = train["ClaimNb"].var()
print(f"raw ClaimNb mean: {raw_mean:.5f}")
print(f"raw ClaimNb var:  {raw_var:.5f}")
print(f"raw var/mean ratio: {raw_var/raw_mean:.4f}")

raw ClaimNb mean: 0.05315
raw ClaimNb var:  0.05654
raw var/mean ratio: 1.0637


The raw (un-modelled) variance-to-mean ratio is only **1.06** - barely above 1, which on its own would suggest almost no overdispersion at all. This isn't a contradiction of the 2.31 figure above; it's a different, less informative question. The raw ratio treats every policy as if it had the same exposure and the same risk, so it can't see the overdispersion that only shows up once you compare each policy's *actual* claims against *its own* fitted expectation (which is what the Pearson chi-square statistic does). This is a reminder that a quick raw check is a useful sanity check, but not a substitute for the model-based diagnostic when exposure and risk both vary across the portfolio - which they do here, substantially.

## 4. Summary of steps 1-2

- Baseline portfolio frequency: **0.1006 claims/policy-year**.
- Poisson GLM (48 parameters, converged) fit on the training split, using every engineered   feature from Phase 4. Total predicted claims matches total observed claims exactly on   training data (25,227.00 vs 25,227) - the expected property of this model class, confirming   it was fit and read correctly once the offset bug above was found and fixed.
- **A real bug was caught and fixed**: statsmodels' raw `.predict()` silently drops the   offset, overstating predictions by 2x+ with no warning. `predict_frequency()` now makes   this mistake structurally impossible, backed by a regression test.
- **Overdispersion is material**: Pearson dispersion ratio 2.31 (the deviance-based ratio,   0.31, is unreliable here because almost all fitted means are far below 1 - a known   limitation of that diagnostic in low-frequency data, not a contradiction of the finding).

### What carries into the next step
Given the material overdispersion found here, the plan's own criterion ("a Negative Binomial model will not be added merely because it is more sophisticated... only when the assumptions and diagnostics justify it") is satisfied: a Negative Binomial challenger is warranted and will be fit and compared in the next step, alongside deciding the final feature set and considering regularization.